# Ensemble Model for Coral Image Classification

This notebook integrates a custom CNN with CBAM and a VGG19-based model using a voting mechanism.

**Task:** Binary classification of bleached vs healthy coral reefs
**Models:** Custom CNN + CBAM | VGG19 (Transfer Learning) | Ensemble (Avg Voting)
**Best Result:** 84% ensemble accuracy

## 1. Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Flatten, Average
from tensorflow.keras.applications import VGG19
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D, Add, Multiply, Conv2D, Input
from tensorflow.keras.optimizers import Adam
import numpy as np

## 2. Data Loading
Update the paths below to point to your local dataset directory.

In [ ]:
# Update these paths to your dataset location
train_dir = 'data/Training'
validation_dir = 'data/Validation'
test_dir = 'data/Testing'

data_gen = ImageDataGenerator(rescale=1.0/255.0)

train_generator = data_gen.flow_from_directory(train_dir, target_size=(128, 128), batch_size=32, class_mode='binary')
validation_generator = data_gen.flow_from_directory(validation_dir, target_size=(128, 128), batch_size=32, class_mode='binary')
test_generator = data_gen.flow_from_directory(test_dir, target_size=(128, 128), batch_size=32, class_mode='binary', shuffle=False)

## 3. Model Definitions

In [ ]:
def cbam_block(input_tensor, reduction_ratio=8):
    channel = input_tensor.shape[-1]
    avg_pool = GlobalAveragePooling2D()(input_tensor)
    max_pool = GlobalMaxPooling2D()(input_tensor)
    shared_dense = Dense(channel // reduction_ratio, activation='relu', use_bias=False)
    avg_out = shared_dense(avg_pool)
    max_out = shared_dense(max_pool)
    channel_attention = Add()([avg_out, max_out])
    channel_attention = Dense(channel, activation='sigmoid')(channel_attention)
    channel_attention = Multiply()([input_tensor, channel_attention])
    spatial_attention = Conv2D(1, kernel_size=7, activation='sigmoid', padding='same')(
        Add()([
            tf.keras.layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(channel_attention),
            tf.keras.layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(channel_attention)
        ])
    )
    return Multiply()([channel_attention, spatial_attention])

def build_cnn_with_cbam(input_shape):
    inputs = Input(shape=input_shape)
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = cbam_block(x)
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = cbam_block(x)
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    outputs = Dense(1, activation='sigmoid')(x)
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_vgg19(input_shape):
    base_model = VGG19(weights='imagenet', include_top=False, input_shape=input_shape)
    for layer in base_model.layers:
        layer.trainable = False
    model = Sequential([base_model, Flatten(), Dense(256, activation='relu'), Dense(1, activation='sigmoid')])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

## 4. Training

In [ ]:
cnn_model = build_cnn_with_cbam((128, 128, 3))
vgg19_model = build_vgg19((128, 128, 3))

cnn_model.fit(train_generator, validation_data=validation_generator, epochs=20)
vgg19_model.fit(train_generator, validation_data=validation_generator, epochs=10)

## 5. Ensemble Prediction & Evaluation

In [ ]:
cnn_predictions = cnn_model.predict(test_generator)
vgg19_predictions = vgg19_model.predict(test_generator)
ensemble_predictions = (cnn_predictions + vgg19_predictions) / 2
final_predictions = (ensemble_predictions > 0.5).astype(int)
true_labels = test_generator.classes
accuracy = np.mean(final_predictions.flatten() == true_labels)
print(f'Ensemble Accuracy: {accuracy * 100:.2f}%')

## 6. Visualizations (ROC, Confusion Matrix, Metrics)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

# Confusion Matrix
cm = confusion_matrix(true_labels, final_predictions.flatten())
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Healthy', 'Bleached'], yticklabels=['Healthy', 'Bleached'])
plt.title('Confusion Matrix - Ensemble Model')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.show()

# Classification Report
print(classification_report(true_labels, final_predictions.flatten(), target_names=['Healthy', 'Bleached']))